# Miniproyecto 1: Clasificación de MRI con Redes Neuronales Convolucionales para Tipos de Cáncer Cerebral

**Universidad de los Andes — Maestría en Inteligencia Artificial (MAIA)**  
**Curso:** Técnicas de Deep Learning (MAIA-4213)

### Autores
| Nombre Completo | Correo Electrónico | Código |
| :--- | :--- | :--- |
| **Otto Wilmer Castillo Santacruz** | `o.castillos@uniandes.edu.co` | 202611101 |
| **Edgar Andrés Pérez Mora** | `ea.perezm1@uniandes.edu.co` | 200822340 |
| **Silvio Yopasa Romero** | `s.yopasa@uniandes.edu.co` | 202615521 |

---

## Contexto
La resonancia magnética (MRI) desempeña un papel fundamental en la detección y clasificación de los diferentes tipos de cáncer cerebral. La aplicación de redes neuronales convolucionales (CNN) ha incrementado notablemente la precisión en la interpretación de estos volúmenes, al ser entrenadas para reconocer patrones y características distintivas asociadas a cada tipo de tumor. Las resonancias magnéticas (MRI) generan volúmenes tridimensionales compuestos por múltiples capas, donde cada una corresponde a una imagen bidimensional del tejido analizado.

Procesar un volumen completo implica manejar una gran cantidad de datos simultáneamente, lo que resulta computacionalmente costoso y demanda considerables recursos de tiempo y memoria. En este trabajo se emplean exclusivamente imágenes individuales extraídas de las capas de los estudios MRI, en lugar de procesar el volumen tridimensional completo. Esta estrategia transforma el problema en una tarea de clasificación supervisada de imágenes bidimensionales, reduciendo significativamente la carga computacional.

### A. Objetivo
- Desarrollar un método basado en redes neuronales convolucionales que permita clasificar con exactitud imágenes extraídas de estudios MRI en una de las 4 categorías diagnósticas (`glioma`, `meningioma`, `notumor`, `pituitary`).

### B. Conjunto de Datos
- El conjunto de datos comprende $7\,023$ imágenes bidimensionales extraídas de resonancias magnéticas cerebrales adquiridas desde distintos planos anatómicos y disponibles en el dominio público.

---
## 0. Configuración del Entorno e Importación de Librerías
Se importan las librerías requeridas para el procesamiento de imágenes, la manipulación de tensores, el entrenamiento de modelos en PyTorch y el cálculo de métricas de desempeño. Asimismo, se fijan las semillas pseudoaleatorias para garantizar la reproducibilidad experimental y se configura el dispositivo de aceleración por hardware (`cuda`, `mps` o `cpu`).

In [ ]:
from pathlib import Path
import pandas as pd


---
## 1. Preparación de las Imágenes para el Entrenamiento y Prueba del Modelo
En esta etapa se estructura el flujo de ingestión, exploración y preprocesamiento de las imágenes de resonancia magnética, garantizando una separación estratificada en subconjuntos de entrenamiento, validación y prueba.

### 1.1. Carga e Indexación del Conjunto de Datos
Se define la ruta de almacenamiento del conjunto de datos, se establecen las 4 categorías diagnósticas (`glioma`, `meningioma`, `notumor`, `pituitary`) y se construye un registro tabular con las rutas de las $7\,023$ imágenes junto con su codificación numérica correspondiente.

In [ ]:


CLASSES = ["glioma", "meningioma", "healthy", "pituitary"]
CLASS_TO_IDX = {cls_name: idx for idx, cls_name in enumerate(CLASSES)}
IDX_TO_CLASS = {idx: cls_name for idx, cls_name in enumerate(CLASSES)}

DATA_DIR = Path("./data/brain-tumor-mri-scans")

def collect_image_paths(root_dir: Path, classes: list) -> pd.DataFrame:
    records = []
    
    for file_path in root_dir.rglob("*"):
        if file_path.is_file():
            parent_name = file_path.parent.name.lower()
            for cls_name in classes:
                if cls_name in parent_name:
                    records.append({
                        "filepath": str(file_path.resolve()),
                        "label_name": cls_name,
                        "label": CLASS_TO_IDX[cls_name]
                    })
                    break
    return pd.DataFrame(records)

df_images = collect_image_paths(DATA_DIR, CLASSES)
print(f"Total de imágenes indexadas: {len(df_images)}")
display(df_images.head())

df_images["label_name"].value_counts()

Total de imágenes indexadas: 7023


,filepath,label_name,label
0,/Users/silvioyopasaromero/Documents/MAC_ANDES_...,pituitary,3
1,/Users/silvioyopasaromero/Documents/MAC_ANDES_...,pituitary,3
2,/Users/silvioyopasaromero/Documents/MAC_ANDES_...,pituitary,3
3,/Users/silvioyopasaromero/Documents/MAC_ANDES_...,pituitary,3
4,/Users/silvioyopasaromero/Documents/MAC_ANDES_...,pituitary,3


label_name
healthy       2000
pituitary     1757
meningioma    1645
glioma        1621
Name: count, dtype: int64

### 1.2. Análisis Exploratorio de Datos (EDA)
Se analiza la distribución de frecuencias absolutas y relativas por categoría diagnóstica con el propósito de identificar posibles desbalances de clase. Adicionalmente, se inspeccionan las dimensiones espaciales de los cortes y se visualiza una muestra representativa de cada clase.

In [ ]:
# Distribución de clases e inspección visual de muestras de MRI


### 1.3. Partición en Conjuntos de Entrenamiento, Validación y Prueba
El conjunto de datos se divide de forma estratificada en los subconjuntos de entrenamiento (`Train`), validación (`Val`) y prueba (`Test`), preservando las proporciones originales de cada categoría y evitando la fuga de información (*data leakage*) entre particiones.

In [ ]:
# Partición estratificada en conjuntos de entrenamiento, validación y prueba


### 1.4. Pipeline de Transformaciones, `Dataset` y `DataLoaders`
Se establecen las transformaciones de estandarización espacial, conversión a tensor y normalización de intensidad. Para el subconjunto de entrenamiento se incorporan técnicas de aumento de datos (*Data Augmentation*) con el fin de favorecer la invariancia espacial del modelo. Posteriormente, se instancian las estructuras `Dataset` y `DataLoader` para el suministro eficiente por lotes (*batches*).

In [ ]:
# Definición de transformaciones, clase Dataset y generadores DataLoader


---
## 2. Desarrollo de la Arquitectura de Red Neuronal Convolucional (CNN)
En esta sección se formula e implementa la arquitectura de red neuronal convolucional orientada a la clasificación de los cortes de MRI en las 4 categorías de estudio, contemplando el uso de bloques convolucionales con reducción espacial (*Max Pooling*) o esquemas de transferencia de aprendizaje (*fine-tuning*) sobre redes preentrenadas.

### 2.1. Definición de la Arquitectura del Modelo
Se construye el modelo convolucional definiendo las capas de extracción de características, los mecanismos de normalización y regularización (`BatchNorm2d`, `Dropout`) y el cabezal de clasificación con dimensión de salida $C = 4$.

In [ ]:
# Definición e instanciación de la arquitectura de red neuronal convolucional


### 2.2. Configuración de Función de Pérdida, Optimizador y Ciclo de Entrenamiento
Se especifican la función de pérdida de entropía cruzada multiclase, el algoritmo de optimización, el planificador de tasa de aprendizaje y las rutinas modulares encargadas de ejecutar cada época de entrenamiento y evaluación sobre el conjunto de validación.

In [ ]:
# Funciones modulares de entrenamiento por época, evaluación y ajuste del modelo


### 2.3. Ejecución del Entrenamiento y Registro de Métricas
Se ejecuta el proceso de optimización del modelo a lo largo de las épocas establecidas, monitoreando la pérdida y el desempeño en entrenamiento y validación, y conservando los pesos asociados al mejor estado de generalización.

In [ ]:
# Ejecución del entrenamiento y almacenamiento del historial de métricas


---
## 3. Evaluación del Modelo y Análisis de Resultados

### 3.1. Curvas de Aprendizaje
Se grafica la evolución temporal de la función de pérdida y la exactitud (*Accuracy* / *F1-Score*) en los subconjuntos de entrenamiento y validación con el objetivo de analizar la estabilidad de la convergencia y diagnosticar posibles comportamientos de subajuste (*underfitting*) o sobreajuste (*overfitting*).

In [ ]:
# Visualización de curvas de pérdida y exactitud durante el entrenamiento


### 3.2. Desempeño Cuantitativo en el Conjunto de Prueba (`Test`)
Se evalúa la capacidad de generalización del modelo final sobre el conjunto de prueba independiente, reportando la exactitud global (*Accuracy*), precisión (*Precision*), sensibilidad (*Recall*), *Macro F1-Score* desagregado por categoría y la matriz de confusión correspondiente.

In [ ]:
# Evaluación cuantitativa en el conjunto de prueba y generación de matriz de confusión


### 3.3. Análisis Cualitativo de Predicciones
Se presenta una inspección visual sobre muestras del conjunto de prueba contrastando la etiqueta clínica real frente a la predicción asignada por la red, incluyendo tanto aciertos como casos de confusión entre patologías.

In [ ]:
# Visualización cualitativa de predicciones correctas y errores de clasificación


---
## 4. Discusión y Conclusiones
- **Síntesis de resultados:** 
- **Análisis por categoría tumoral:** 
- **Limitaciones y trabajo futuro:** 